In [ ]:
import os
import re
import json
import copy
import numpy as np
import torch
import matplotlib.pyplot as plt

import datasets
import training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR = "./experiment_results"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
DATA_FOLDER = "../output_quan"   # folder containing both Image-...jpg and State-...mat
TRAIN_SIMS_PATH = "../train_sims.npy"
VAL_SIMS_PATH   = "../val_sims.npy"
TEST_SIMS_PATH  = "../test_sims.npy"

MAX_SIM_ID = 300          # because you said you have about 300 sims
MIN_EPOCHS = 3
MAX_EPOCHS = 10
VAL_STEPS = (1, 201)

# if your data is only one folder, keep types=[0] and point both folder args at same folder
BASE_DATASET_ARGS = {
    "points_per_side": 3,
    "radius": 5,
    "steps": (0, 200),
    "types": [0],
    "H": 200,
    "W": 200,
    "channels": "all",
    "future_delta": 0,
    "binary_folder": DATA_FOLDER,
    "uniform_folder": DATA_FOLDER,
}

In [ ]:
# Representations to compare
EXPERIMENTS = [
    {
        "name": "jpg_raw255",
        "dataset_args": {
            "data_source": "jpg",
            "jpg_scale": "raw255",
        },
    },
    {
        "name": "jpg_01",
        "dataset_args": {
            "data_source": "jpg",
            "jpg_scale": "01",
        },
    },
    {
        "name": "jpg_neg11",
        "dataset_args": {
            "data_source": "jpg",
            "jpg_scale": "neg11",
        },
    },
    {
        "name": "mat_raw",
        "dataset_args": {
            "data_source": "mat",
            "mat_variant": "raw",
            "mat_scale": None,
        },
    },
    {
        "name": "mat_255",
        "dataset_args": {
            "data_source": "mat",
            "mat_variant": "255",
            "mat_scale": None,
        },
    },
    {
        "name": "mat_u8",
        "dataset_args": {
            "data_source": "mat",
            "mat_variant": "u8",
            "mat_scale": None,
        },
    },
    {
        "name": "mat_255_neg11",
        "dataset_args": {
            "data_source": "mat",
            "mat_variant": "255",
            "mat_scale": "neg11",
        },
    },
]

In [ ]:
# HELPERS
# =========================
def get_available_sims(folder: str) -> np.ndarray:
    """
    Finds sims that exist in the folder by looking for State-<sim>-<step>.mat or Image-<sim>-<step>_P.jpg
    """
    sim_ids = set()

    state_pat = re.compile(r"^State-(\d+)-\d+\.mat$")
    img_pat = re.compile(r"^Image-(\d+)-\d+_P\.jpg$")

    for fname in os.listdir(folder):
        m1 = state_pat.match(fname)
        if m1:
            sim_ids.add(int(m1.group(1)))
            continue
        m2 = img_pat.match(fname)
        if m2:
            sim_ids.add(int(m2.group(1)))

    return np.array(sorted(sim_ids), dtype=int)


def filter_split(split: np.ndarray, available_sims: np.ndarray, max_sim_id: int) -> np.ndarray:
    return split[np.isin(split, available_sims) & (split <= max_sim_id)]


def fallback_split(available_sims: np.ndarray):
    """
    If the provided npy splits don't work well with the current folder, make simple splits.
    """
    sims = np.array(sorted(available_sims))
    n = len(sims)
    n_train = max(1, int(0.7 * n))
    n_val = max(1, int(0.15 * n))

    train = sims[:n_train]
    val = sims[n_train:n_train + n_val]
    test = sims[n_train + n_val:]
    return train, val, test


def load_and_filter_splits():
    available_sims = get_available_sims(DATA_FOLDER)
    available_sims = available_sims[available_sims <= MAX_SIM_ID]

    print(f"Found {len(available_sims)} available sims in folder up to sim {MAX_SIM_ID}.")

    train_sims = np.load(TRAIN_SIMS_PATH)
    val_sims = np.load(VAL_SIMS_PATH)
    test_sims = np.load(TEST_SIMS_PATH)

    train_sims = filter_split(train_sims, available_sims, MAX_SIM_ID)
    val_sims = filter_split(val_sims, available_sims, MAX_SIM_ID)
    test_sims = filter_split(test_sims, available_sims, MAX_SIM_ID)

    # If any split is empty, make fallback splits from available sims
    if len(train_sims) == 0 or len(val_sims) == 0:
        print("Provided split files do not match current available sims well. Using fallback 70/15/15 split.")
        train_sims, val_sims, test_sims = fallback_split(available_sims)

    print(f"Train sims: {len(train_sims)}")
    print(f"Val sims:   {len(val_sims)}")
    print(f"Test sims:  {len(test_sims)}")

    return train_sims, val_sims, test_sims


def run_experiment(name: str, dataset_args: dict, train_sims, val_sims):
    args = copy.deepcopy(BASE_DATASET_ARGS)
    args.update(dataset_args)

    print(f"\n==== Running {name} ====")
    print(json.dumps(args, indent=2, default=str))

    model, train_losses, val_losses = training.train_from_scratch(
        dataset_type=datasets.FixedDenseDatasetFull,
        dataset_arguments=args,
        train_sims=train_sims,
        val_sims=val_sims,
        optimizer=lambda params: torch.optim.Adam(params, lr=1e-3),
        min_epochs=MIN_EPOCHS,
        max_epochs=MAX_EPOCHS,
        val_steps=VAL_STEPS,
        device=device,
        physics_fn=training.physics_none,   # change if you want, e.g. training.physics_full
    )

    out = {
        "name": name,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "final_train_loss": float(train_losses[-1]) if train_losses else None,
        "final_val_loss": float(val_losses[-1]) if val_losses else None,
    }

    # save model
    torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{name}.pt"))

    # save metrics
    with open(os.path.join(SAVE_DIR, f"{name}_metrics.json"), "w") as f:
        json.dump(out, f, indent=2)

    return out


def plot_results(results):
    plt.figure(figsize=(10, 6))
    for r in results:
        if r["val_losses"]:
            plt.plot(r["val_losses"], label=r["name"])
    plt.title("Validation Loss by Data Representation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "val_loss_comparison.png"), dpi=200)
    plt.show()

In [ ]:
# MAIN
# =========================
def main():
    train_sims, val_sims, test_sims = load_and_filter_splits()

    all_results = []
    for exp in EXPERIMENTS:
        result = run_experiment(
            name=exp["name"],
            dataset_args=exp["dataset_args"],
            train_sims=train_sims,
            val_sims=val_sims,
        )
        all_results.append(result)

    # save summary
    summary_path = os.path.join(SAVE_DIR, "summary.json")
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=2)

    print("\n==== Final Summary ====")
    for r in all_results:
        print(
            f"{r['name']:15s} | "
            f"final_train={r['final_train_loss']:.6g} | "
            f"final_val={r['final_val_loss']:.6g}"
        )

    plot_results(all_results)


if __name__ == "__main__":
    main()